# EPIC Clarity Observation Hydration

This notebook hydrates the OMOP OBSERVATION table from EPIC Clarity patient data for non-tabular facts.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_4` - Gender identity and extended patient attributes
- `_exponent._bronze_epic_clarity_*.dbo_SOCIAL_HX` - Social history (tobacco use, etc.)
- `_exponent._bronze_epic_clarity_*.dbo_PAT_ENC_RSN_VISIT` - Reason for visit/encounter reason

## OMOP Fields Populated
- observation_id (surrogate key)
- observation_source_value
- observation_concept_id (mapped from observation type)
- observation_date
- value_source_value
- value_as_concept_id (if value maps to concept)

## Notes
- Captures gender identity separately from sex at birth (person table)
- Social history facts like tobacco use
- Encounter reason/chief complaint

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_observation_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'patient_3', 'GENDER_IDENTITY', p4.PAT_ID) AS observation_source_value,
    'Gender Identity' AS observation_type_source_value,
    p4.GENDER_IDENTITY_C_NAME AS value_source_value,
    NULL AS observation_date,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.patient_3 p4
WHERE p4.GENDER_IDENTITY_C_NAME IS NOT NULL
''')

display(silver_observation_df)
silver_observation_df.createOrReplaceTempView("silver_observation")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.observation AS target
USING silver_observation AS source
ON target.observation_source_value = source.observation_source_value

WHEN MATCHED AND NOT (
    target.observation_type_source_value <=> source.observation_type_source_value
    AND target.value_source_value <=> source.value_source_value
    AND target.observation_date <=> source.observation_date
)
THEN UPDATE SET
    target.observation_type_source_value = source.observation_type_source_value,
    target.value_source_value = source.value_source_value,
    target.observation_date = source.observation_date,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    observation_source_value,
    observation_type_source_value,
    value_source_value,
    observation_date,
    updated_tsp
)
VALUES (
    source.observation_source_value,
    source.observation_type_source_value,
    source.value_source_value,
    source.observation_date,
    source.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
    source_system,
    observation_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.observation_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, observation_source_value, updated_tsp
    FROM _exponent.omop_silver.observation
    WHERE observation_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation x
    ON s.observation_source_value = x.observation_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
gold_observation_df = spark.sql("""
SELECT
    m.observation_id,
    s.observation_date,
    s.value_source_value,
    COALESCE(ocm.concept_id, 0) AS observation_concept_id,
    COALESCE(vcm.concept_id, 0) AS value_as_concept_id,
    s.updated_tsp
FROM _exponent.omop_silver.observation s
INNER JOIN _exponent.omop_mapping.source_to_observation m
    ON s.observation_source_value = m.observation_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept ocm
    ON ocm.source_id = s.observation_type_source_value
    AND ocm.domain_id = 'Observation'
    AND ocm.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept vcm
    ON vcm.source_id = s.value_source_value
    AND vcm.source_system = 'epic_clarity'
""")

display(gold_observation_df)
gold_observation_df.createOrReplaceTempView("gold_observation")

In [ ]:
%sql
MERGE INTO _exponent.omop.observation AS target
USING gold_observation AS source
ON target.observation_id = source.observation_id

WHEN MATCHED AND NOT (
    target.observation_concept_id <=> source.observation_concept_id
    AND target.observation_date <=> source.observation_date
    AND target.value_source_value <=> source.value_source_value
    AND target.value_as_concept_id <=> source.value_as_concept_id
)
THEN UPDATE SET
    target.observation_concept_id = source.observation_concept_id,
    target.observation_date = source.observation_date,
    target.value_source_value = source.value_source_value,
    target.value_as_concept_id = source.value_as_concept_id

WHEN NOT MATCHED THEN INSERT (
    observation_id,
    observation_concept_id,
    observation_date,
    value_source_value,
    value_as_concept_id
)
VALUES (
    source.observation_id,
    source.observation_concept_id,
    source.observation_date,
    source.value_source_value,
    source.value_as_concept_id
)